In [4]:
import os
import json
import re
from typing import Dict, List, Optional
from pathlib import Path
from dotenv import load_dotenv
from llama_parse import LlamaParse

# LangChain components
from langchain.document_loaders import JSONLoader
from langchain.retrievers import ParentDocumentRetriever
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.storage import InMemoryStore
from langchain.schema import Document
from langchain_google_genai import ChatGoogleGenerativeAI

# Load environment variables
load_dotenv()

class PDFProcessor:
    """Handles PDF parsing with LlamaParse and saves to Markdown"""
    
    def __init__(self, output_dir: str = "markdown_output"):
        self.parser = LlamaParse(
            api_key=os.getenv("LLAMA_PARSE_KEY"),
            result_type="markdown",
            verbose=True,
        )
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)
        
    def parse_pdf_to_markdown(self, pdf_path: str) -> Optional[str]:
        """Process PDF and save as markdown file"""
        try:
            if not os.path.exists(pdf_path):
                raise FileNotFoundError(f"PDF file not found: {pdf_path}")
                
            print(f"Processing PDF: {pdf_path}")
            documents = self.parser.load_data(pdf_path)
            markdown_text = "\n".join(doc.text for doc in documents)
            
            # Save to markdown file
            base_name = os.path.splitext(os.path.basename(pdf_path))[0]
            output_path = os.path.join(self.output_dir, f"{base_name}.md")
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(markdown_text)
                
            return output_path  # Return path to saved markdown file
        except Exception as e:
            print(f"Error processing {pdf_path}: {e}")
            return None

class RAGSystem:
    """RAG System that works with saved Markdown files"""
    
    def __init__(self):
        # Initialize components
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1500,
            chunk_overlap=300,
            separators=["\n\n", "\n", ".", " "]
        )
        
        self.embedding_model = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        
        # Initialize storage
        self.vectorstore = None
        self.docstore = InMemoryStore()
        self.retriever = None
        
    def load_markdown_file(self, md_path: str, metadata: dict = {}) -> List[Document]:
        """Load and split a markdown file into documents"""
        try:
            with open(md_path, 'r', encoding='utf-8') as f:
                markdown_text = f.read()
            
            doc = Document(
                page_content=markdown_text,
                metadata={"source": md_path, **metadata}
            )
            return self.text_splitter.split_documents([doc])
        except Exception as e:
            print(f"Error loading markdown file: {e}")
            return []
    
    def initialize_retriever(self, documents: List[Document]):
        """Initialize the retriever with documents"""
        self.vectorstore = Chroma.from_documents(documents, self.embedding_model)
        self.retriever = ParentDocumentRetriever(
            vectorstore=self.vectorstore,
            docstore=self.docstore,
            child_splitter=self.text_splitter
        )
        self.retriever.add_documents(documents)
    
    def retrieve_relevant_documents(self, query: str, top_k: int = 3) -> str:
        """Retrieve relevant document chunks for a query"""
        try:
            if not self.retriever:
                raise ValueError("Retriever not initialized. Please add documents first.")
                
            results = self.retriever.invoke(query)
            context = "\n\n".join([doc.page_content for doc in results[:top_k]])
            return context
        except Exception as e:
            return f"Error during retrieval: {str(e)}"
    
    def json_to_obj(self, json_str: str) -> dict:
        """Clean and parse JSON output from LLM"""
        cleaned = re.sub(r"^```json\s*|\s*```$", "", json_str.strip(), flags=re.IGNORECASE)
        try:
            return json.loads(cleaned)
        except json.JSONDecodeError as e:
            print("Failed to parse JSON: ", e)
            print("Raw content was:", json_str)
            return {"answer": json_str}

class PDFQASystem:
    """System that uses saved Markdown files for QA and summarization"""
    
    def __init__(self):
        self.pdf_processor = PDFProcessor()
        self.rag_system = RAGSystem()
        self.loaded_documents = []
        self.llm = ChatGoogleGenerativeAI(
            model="gemini-1.5-flash",
            temperature=0.7,
            api_key=os.getenv("GEMINI_API_KEY"),
        )
    
    def process_pdf_and_save_markdown(self, pdf_path: str) -> Optional[str]:
        """Process a PDF file and save as markdown"""
        return self.pdf_processor.parse_pdf_to_markdown(pdf_path)
    
    def load_markdown_file(self, md_path: str, metadata: dict = {}) -> bool:
        """Load a markdown file into the system"""
        try:
            documents = self.rag_system.load_markdown_file(md_path, metadata)
            if not documents:
                return False
                
            self.loaded_documents.extend(documents)
            
            if not self.rag_system.retriever:
                self.rag_system.initialize_retriever(documents)
            else:
                self.rag_system.retriever.add_documents(documents)
                
            return True
        except Exception as e:
            print(f"Error loading markdown file: {e}")
            return False
    
    def ask_question(self, question: str) -> Dict[str, str]:
        """Your original QA prompt"""
        if not self.loaded_documents:
            return {"question": question, "answer": "No documents loaded. Please load markdown files first."}
            
        QA_PROMPT = """  
        You are a helpful assistant that answers questions based strictly on the provided context.
        Answer the question clearly and concisely using only the information from the context.
        If the context doesn't contain the answer, say "I don't know."

        Context:
        {context}

        Question: {question}

        Please respond in JSON format with 'question' and 'answer' fields.
        """
        
        context = self.rag_system.retrieve_relevant_documents(question)
        prompt = QA_PROMPT.format(context=context, question=question)
        response = self.llm.invoke(prompt)
        return self.rag_system.json_to_obj(response.content)
    
    def generate_concise_summary(self) -> Dict[str, str]:
        """Your original summary prompt"""
        if not self.loaded_documents:
            return {"error": "No documents loaded for summarization"}
            
        full_text = "\n\n".join([doc.page_content for doc in self.loaded_documents])
        
        SUMMARY_PROMPT = """
        Task:
Analyze the provided PDF document and generate a structured summary that captures key information clearly and concisely. Adapt the output format based on the document type (technical, legal, course material, article, etc.).

Output Guidelines:
Title & Main Topic:

Identify the document's primary subject (e.g., "Microcontrollers," "Legal Contract Terms").

Summarize the core purpose/theme in 1–2 sentences.

Key Sections/Components:

Break down the document into logical subtopics (headings, chapters, or themes).

For each subtopic, list bullet points (1–2 lines each) covering:

Definitions (if technical).

Critical features/functions (for technical docs).

Key arguments/findings (for articles/research).

Clauses/obligations (for legal docs).

Applications/Examples (if applicable):

Highlight real-world uses, case studies, or scenarios.

Comparative Analysis (if relevant):

Contrast concepts (e.g., "Interrupts vs. Polling," "Analog vs. Digital Signals").

Technical/Legal Nuances:

For technical docs: Note specs (e.g., voltage ranges, memory types).

For legal docs: Summarize obligations, restrictions, penalties.


Formatting Rules:
Use bold headers for main topics (###).

Bullet points for brevity (-).

Italics for emphasis or definitions (*term*).

Tables/figures only if referenced in the text.

Examples of Adaptability:

Technical PDF (e.g., Microcontrollers): Focus on components, specs, applications.

Course/Lesson PDF: Summarize modules, key learnings, exercises.

Legal PDF: Extract clauses, parties, deadlines, penalties.

Article/Research PDF: Highlight thesis, methodology, conclusions.
        {context}
        """
        
        prompt = SUMMARY_PROMPT.format(context=full_text)
        response = self.llm.invoke(prompt)
        return {"summary": response.content}

def interactive_qa_session():
    """Interactive session that first saves markdown, then uses it"""
    system = PDFQASystem()
    
    # Step 1: Process PDF and save markdown
    pdf_path = input("Enter PDF file path to process: ").strip()
    md_path = system.process_pdf_and_save_markdown(pdf_path)
    
    if not md_path:
        print("Failed to process PDF")
        return
    
    print(f"Markdown file saved to: {md_path}")
    
    # Step 2: Load the markdown file
    if not system.load_markdown_file(md_path, {"title": os.path.basename(md_path)}):
        print("Failed to load markdown file")
        return
    
    # Main interaction loop
    while True:
        print("\nOptions:")
        print("1. Question Answering")
        print("2. Generate Summary")
        print("3. Exit")
        
        choice = input("\nEnter choice (1-3): ").strip()
        
        if choice == "1":
            print("\nQuestion Answering Mode (type 'back' to return)")
            while True:
                question = input("\nYour question: ").strip()
                if question.lower() == 'back':
                    break
                answer = system.ask_question(question)
                print(f"\nAnswer: {answer['answer']}")
                
        elif choice == "2":
            summary = system.generate_concise_summary()
            print("\n=== Document Summary ===")
            print(summary["summary"])
            print("=======================")
            
        elif choice == "3":
            print("Goodbye!")
            break
            
        else:
            print("Invalid choice")

if __name__ == "__main__":
    interactive_qa_session()

Processing PDF: data1.pdf
Started parsing the file under job_id 1ce5dd7f-78fa-4fbe-9b91-6ae2f1ab8669
Markdown file saved to: markdown_output\data1.md

Options:
1. Question Answering
2. Generate Summary
3. Exit
Goodbye!
